# Notebook 9 — Entity extraction as a recovery metric

**Why a new metric.** MMLU measures specific factual knowledge stored in MLP layers. Pruning MLP tiles directly attacks the fact-storage substrate (Geva et al. 2020). C4-style recovery training can restore *general circuits* but cannot restore *specific facts* not present in the recovery corpus. So MMLU has been an unfair judge of LoRA recovery — it's measuring the part of capability that's hardest to restore without teacher distillation.

**The cleaner metric.** Entity extraction (CoNLL-2003 NER) measures pattern-level language understanding without requiring deep factual recall. Recognizing that 'Angela Merkel' is a person and 'Berlin' is a location is about syntactic + linguistic structure — exactly the capability we expect general-circuit recovery to restore.

**This notebook does:**
1. Load dense Qwen 2.5 3B as teacher.
2. Load CoNLL-2003 dev split.
3. Run teacher on the dataset, measure: (a) perplexity on the entity-annotation tokens, (b) entity F1 via greedy decoding on a subset.
4. **Cache top-K=64 teacher logits to disk** during the teacher forward pass — for offline distillation in notebook 10. ~36 MB on disk.
5. Apply Taylor 10% pruning, re-run all metrics on the pruned student.
6. Report the gap.

**Hypothesis.** Dense will get reasonable F1 (literature suggests ~70-80% for instruct models with few-shot NER). Taylor 10% pruning will degrade it by some amount X. Notebook 10's distillation should close most of X if the technique works as theory predicts.

In [1]:
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

import gc, json, math, re, time
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from tqdm import tqdm
from datasets import load_dataset
from transformers import AutoModelForCausalLM, AutoTokenizer

import config

MODEL_NAME = "Qwen/Qwen2.5-3B-Instruct"
TILE_R, TILE_C = config.TILE_SIZES[0]
PRUNE_SPARSITY = 0.10

N_PERPLEXITY_SAMPLES = 500
N_F1_SAMPLES = 200
TOP_K = 64
MAX_NEW_TOKENS = 64

TEACHER_CACHE_PATH = "results/teacher_conll_logits.pt"

torch.set_float32_matmul_precision("high")
free, total = torch.cuda.mem_get_info()
print(f"GPU free: {free/1e9:.2f} / {total/1e9:.2f} GB")

GPU free: 7.59 / 8.18 GB


## 1. Load model + cache MLP weights for un-prune

In [2]:
print(f"loading {MODEL_NAME}...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=getattr(torch, config.TORCH_DTYPE),
    device_map=config.DEVICE,
)
model.eval()
model.config.use_cache = True  # required for generate()

def is_mlp_name(name):
    return any(k in name for k in config.PRUNE_TARGETS_PATTERNS)

ORIGINAL_MLP_WEIGHTS = {}
for name, module in model.named_modules():
    if isinstance(module, nn.Linear) and is_mlp_name(name):
        ORIGINAL_MLP_WEIGHTS[name + ".weight"] = module.weight.data.detach().clone().cpu()

print(f"params: {sum(p.numel() for p in model.parameters())/1e9:.2f}B")
print(f"cached {len(ORIGINAL_MLP_WEIGHTS)} MLP matrices on CPU")
print(f"peak GPU: {torch.cuda.max_memory_allocated()/1e9:.2f} GB")

loading Qwen/Qwen2.5-3B-Instruct...


`torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

params: 3.09B
cached 108 MLP matrices on CPU
peak GPU: 6.22 GB


## 2. CoNLL-2003 dev split + few-shot prompt formatting

We frame NER as in-context learning: 3 fixed few-shot examples covering all 4 entity types (PER/ORG/LOC/MISC), then the test sentence. Model continues with the entity list.

In [3]:
ds = load_dataset("eriktks/conll2003", split="validation", trust_remote_code=True)
print(f"CoNLL-2003 dev: {len(ds)} sentences")
print(f"first example: {ds[0]}")

TAG_ID_TO_NAME = {
    0: "O",
    1: "B-PER", 2: "I-PER",
    3: "B-ORG", 4: "I-ORG",
    5: "B-LOC", 6: "I-LOC",
    7: "B-MISC", 8: "I-MISC",
}

def entities_from_bio(tokens, tag_ids):
    """Convert BIO-tagged tokens to entity (text, type) tuples."""
    entities = []
    cur_words = None
    cur_type = None
    for tok, tid in zip(tokens, tag_ids):
        tag = TAG_ID_TO_NAME[tid]
        if tag.startswith("B-"):
            if cur_words:
                entities.append((" ".join(cur_words), cur_type))
            cur_words = [tok]
            cur_type = tag[2:]
        elif tag.startswith("I-") and cur_words and cur_type == tag[2:]:
            cur_words.append(tok)
        else:  # O or mismatched I-
            if cur_words:
                entities.append((" ".join(cur_words), cur_type))
                cur_words = None
                cur_type = None
    if cur_words:
        entities.append((" ".join(cur_words), cur_type))
    return entities

def format_entities(entities):
    if not entities:
        return "none"
    return ", ".join(f"{txt} ({typ})" for txt, typ in entities)

# Sanity: extract entities from first 3 examples
for i in range(3):
    ents = entities_from_bio(ds[i]["tokens"], ds[i]["ner_tags"])
    print(f"  ex {i}: tokens={' '.join(ds[i]['tokens'][:12])}{'...' if len(ds[i]['tokens'])>12 else ''}")
    print(f"         entities={format_entities(ents)}")

CoNLL-2003 dev: 3250 sentences
first example: {'id': '0', 'tokens': ['CRICKET', '-', 'LEICESTERSHIRE', 'TAKE', 'OVER', 'AT', 'TOP', 'AFTER', 'INNINGS', 'VICTORY', '.'], 'pos_tags': [22, 8, 22, 22, 15, 22, 22, 22, 22, 21, 7], 'chunk_tags': [11, 0, 11, 12, 13, 11, 12, 12, 12, 12, 0], 'ner_tags': [0, 0, 3, 0, 0, 0, 0, 0, 0, 0, 0]}
  ex 0: tokens=CRICKET - LEICESTERSHIRE TAKE OVER AT TOP AFTER INNINGS VICTORY .
         entities=LEICESTERSHIRE (ORG)
  ex 1: tokens=LONDON 1996-08-30
         entities=LONDON (LOC)
  ex 2: tokens=West Indian all-rounder Phil Simmons took four for 38 on Friday as...
         entities=West Indian (MISC), Phil Simmons (PER), Leicestershire (ORG), Somerset (ORG)


In [4]:
# Manually crafted few-shot examples covering all 4 entity types.
# Kept fixed across all evals so any A/B difference reflects model capability, not prompt drift.
FEW_SHOT = [
    ("Angela Merkel visited Berlin last Friday .",
     [("Angela Merkel", "PER"), ("Berlin", "LOC")]),
    ("Apple Inc. acquired Beats Electronics in 2014 .",
     [("Apple Inc.", "ORG"), ("Beats Electronics", "ORG")]),
    ("The Olympic Games will return to Paris in 2024 .",
     [("Olympic Games", "MISC"), ("Paris", "LOC")]),
]

FEW_SHOT_PREFIX = "Extract named entities from each text. Answer with comma-separated 'entity (TYPE)' pairs, where TYPE is PER, ORG, LOC, or MISC. If there are no entities, write 'none'.\n\n"
for text, ents in FEW_SHOT:
    FEW_SHOT_PREFIX += f"Text: {text}\nEntities: {format_entities(ents)}\n\n"

def make_prompt(tokens):
    return FEW_SHOT_PREFIX + f"Text: {' '.join(tokens)}\nEntities:"

def make_full(tokens, entities):
    """Prompt + ' answer\\n' — answer is what we measure perplexity on."""
    return make_prompt(tokens) + " " + format_entities(entities)

# Show a sample full prompt
ex0 = ds[0]
ents0 = entities_from_bio(ex0["tokens"], ex0["ner_tags"])
print("=== sample full prompt ===")
print(make_full(ex0["tokens"], ents0))
print("==========================")

=== sample full prompt ===
Extract named entities from each text. Answer with comma-separated 'entity (TYPE)' pairs, where TYPE is PER, ORG, LOC, or MISC. If there are no entities, write 'none'.

Text: Angela Merkel visited Berlin last Friday .
Entities: Angela Merkel (PER), Berlin (LOC)

Text: Apple Inc. acquired Beats Electronics in 2014 .
Entities: Apple Inc. (ORG), Beats Electronics (ORG)

Text: The Olympic Games will return to Paris in 2024 .
Entities: Olympic Games (MISC), Paris (LOC)

Text: CRICKET - LEICESTERSHIRE TAKE OVER AT TOP AFTER INNINGS VICTORY .
Entities: LEICESTERSHIRE (ORG)


## 3. Eval functions — perplexity on annotation tokens, F1 from generation

In [5]:
def parse_entities_from_completion(text):
    """Parse 'Angela Merkel (PER), Berlin (LOC)' or 'none' into entity tuples."""
    text = text.strip()
    # Strip trailing newline-or-period content if model rambled
    text = text.split("\n")[0].strip()
    if not text or text.lower().startswith("none"):
        return []
    entities = []
    for part in text.split(","):
        part = part.strip()
        m = re.match(r"^(.+?)\s*\(([A-Z]+)\)\s*$", part)
        if m:
            entities.append((m.group(1).strip(), m.group(2).strip()))
    return entities

def micro_f1(pred_lists, ref_lists):
    """Micro-F1 over entity (text, type) tuples across all examples."""
    tp = fp = fn = 0
    for pred, ref in zip(pred_lists, ref_lists):
        pred_set = set(pred)
        ref_set = set(ref)
        tp += len(pred_set & ref_set)
        fp += len(pred_set - ref_set)
        fn += len(ref_set - pred_set)
    p = tp / (tp + fp) if (tp + fp) > 0 else 0.0
    r = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    f1 = 2 * p * r / (p + r) if (p + r) > 0 else 0.0
    return p, r, f1, tp, fp, fn

def eval_perplexity_and_cache(model, tokenizer, examples, cache_topk=False, k=64, tag=""):
    """For each example, compute CE loss only on the entity-annotation tokens.
    Optionally cache top-K teacher logits at those positions for distillation."""
    total_nll = 0.0
    total_tokens = 0
    cache = {} if cache_topk else None
    skipped = 0

    for i, ex in enumerate(tqdm(examples, desc=f"perplexity[{tag}]")):
        toks = ex["tokens"]
        ents = entities_from_bio(toks, ex["ner_tags"])
        prompt = make_prompt(toks)
        full = make_full(toks, ents)
        prompt_ids = tokenizer(prompt, return_tensors="pt")["input_ids"][0]
        full_ids = tokenizer(full, return_tensors="pt")["input_ids"][0]
        if len(full_ids) <= len(prompt_ids):
            skipped += 1
            continue

        with torch.no_grad():
            logits = model(full_ids.unsqueeze(0).to(config.DEVICE)).logits[0]  # (seq, vocab)

        # Position p predicts token p+1. Annotation tokens occupy [len(prompt_ids), len(full_ids)-1].
        # So the predictive positions are [len(prompt_ids)-1, len(full_ids)-2].
        ans_pred_start = len(prompt_ids) - 1
        ans_pred_end = len(full_ids) - 1   # exclusive
        target_ids = full_ids[len(prompt_ids):].to(config.DEVICE)  # tokens being predicted
        ans_logits = logits[ans_pred_start:ans_pred_end]            # (ans_len, vocab)

        log_probs = F.log_softmax(ans_logits.float(), dim=-1)
        nlls = -log_probs.gather(1, target_ids.unsqueeze(1)).squeeze(1)  # (ans_len,)
        total_nll += nlls.sum().item()
        total_tokens += nlls.numel()

        if cache_topk:
            tk_vals, tk_idx = ans_logits.topk(k, dim=-1)
            cache[i] = {
                "input_ids":     full_ids.cpu(),
                "prompt_len":    int(len(prompt_ids)),
                "topk_indices":  tk_idx.cpu().to(torch.int32),
                "topk_logits":   tk_vals.cpu().to(torch.bfloat16),
                "target_ids":    target_ids.cpu(),
            }

    avg_nll = total_nll / max(total_tokens, 1)
    ppl = math.exp(avg_nll)
    return {"perplexity": ppl, "nll": avg_nll, "tokens": total_tokens, "skipped": skipped, "cache": cache}

def eval_f1(model, tokenizer, examples, max_new=MAX_NEW_TOKENS, tag=""):
    """Generate entity list for each example with greedy decoding, parse, compute micro-F1."""
    preds, refs = [], []
    for ex in tqdm(examples, desc=f"F1[{tag}]"):
        toks = ex["tokens"]
        ref = entities_from_bio(toks, ex["ner_tags"])
        prompt = make_prompt(toks)
        prompt_ids = tokenizer(prompt, return_tensors="pt")["input_ids"].to(config.DEVICE)
        with torch.no_grad():
            out = model.generate(
                prompt_ids,
                max_new_tokens=max_new,
                do_sample=False,
                pad_token_id=tokenizer.eos_token_id,
            )
        completion = tokenizer.decode(out[0][prompt_ids.shape[1]:], skip_special_tokens=True)
        preds.append(parse_entities_from_completion(completion))
        refs.append(ref)

    p, r, f1, tp, fp, fn = micro_f1(preds, refs)
    return {"precision": p, "recall": r, "f1": f1, "tp": tp, "fp": fp, "fn": fn,
            "sample_predictions": preds[:5], "sample_references": refs[:5]}

## 4. Build subsets and apply Taylor 10% mask utility

In [6]:
# Deterministic subsets
rng = np.random.RandomState(42)
all_indices = np.arange(len(ds))
ppl_indices = rng.choice(all_indices, size=N_PERPLEXITY_SAMPLES, replace=False)
f1_indices  = rng.choice(all_indices, size=N_F1_SAMPLES,         replace=False)

ppl_examples = [ds[int(i)] for i in ppl_indices]
f1_examples  = [ds[int(i)] for i in f1_indices]
print(f"perplexity subset: {len(ppl_examples)} sentences")
print(f"F1 subset:         {len(f1_examples)} sentences")

perplexity subset: 500 sentences
F1 subset:         200 sentences


In [7]:
# C4 calibration for Taylor (same as notebooks 5/6/7/8)
N_CAL = 512
SEQ_LEN_CAL = 128

cal_ds = load_dataset(config.CALIBRATION_DATASET, config.CALIBRATION_SUBSET,
                     split="train", streaming=True, trust_remote_code=True)
cal_texts = []
for ex in cal_ds:
    t = ex.get("text", "")
    if len(t) > 50:
        cal_texts.append(t)
    if len(cal_texts) >= N_CAL:
        break
cal_encodings = [tokenizer(t, return_tensors="pt", max_length=SEQ_LEN_CAL, truncation=True)
                 for t in cal_texts]
print(f"calibration samples: {len(cal_encodings)}")

Resolving data files:   0%|          | 0/1024 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/1024 [00:00<?, ?it/s]

calibration samples: 512


In [8]:
def get_component_type(name):
    for p in config.PRUNE_TARGETS_PATTERNS:
        if p in name:
            return p
    return "unknown"

def compute_taylor_importance(model, cal_encodings, tag=""):
    for p in model.parameters():
        p.requires_grad_(False)
    for name, p in model.named_parameters():
        if is_mlp_name(name) and p.ndim == 2:
            p.requires_grad_(True)
    model.gradient_checkpointing_enable()
    model.config.use_cache = False

    importance_gpu = {}
    def make_hook(pname):
        def hook(param):
            if param.grad is None:
                return
            taylor = (param.data * param.grad).abs().float()
            out_dim, in_dim = taylor.shape
            nr, nc = out_dim // TILE_R, in_dim // TILE_C
            tt = taylor[:nr*TILE_R, :nc*TILE_C].reshape(nr, TILE_R, nc, TILE_C).permute(0, 2, 1, 3)
            tile_scores = tt.reshape(nr, nc, -1).sum(dim=-1)
            if pname in importance_gpu:
                importance_gpu[pname] += tile_scores
            else:
                importance_gpu[pname] = tile_scores.clone()
            param.grad = None
        return hook

    handles = []
    for name, p in model.named_parameters():
        if is_mlp_name(name) and p.ndim == 2 and p.requires_grad:
            handles.append(p.register_post_accumulate_grad_hook(make_hook(name)))

    n_samples = 0
    for enc in tqdm(cal_encodings, desc=f"taylor[{tag}]"):
        model.zero_grad(set_to_none=True)
        inputs = {k: v.to(config.DEVICE) for k, v in enc.items()}
        out = model(**inputs, labels=inputs["input_ids"])
        out.loss.backward()
        n_samples += 1

    for h in handles:
        h.remove()
    model.gradient_checkpointing_disable()
    model.config.use_cache = True
    for p in model.parameters():
        p.requires_grad_(False)

    importance = {name: (v / n_samples).cpu().numpy() for name, v in importance_gpu.items()}
    del importance_gpu
    gc.collect(); torch.cuda.empty_cache()
    return importance

def apply_taylor_masks_at(importance, sparsity):
    # per-component-type z-norm
    comp_stats = {}
    for ct in config.PRUNE_TARGETS_PATTERNS:
        vals = np.concatenate([m.ravel() for n, m in importance.items() if ct in n])
        comp_stats[ct] = (vals.mean(), vals.std())
    importance_norm = {}
    for name, m in importance.items():
        mu, sd = comp_stats[get_component_type(name)]
        importance_norm[name] = (m - mu) / sd if sd > 1e-8 else np.zeros_like(m)

    masks = {}
    for name, m in importance_norm.items():
        thr = float(np.percentile(m.ravel(), sparsity * 100))
        masks[name] = m < thr

    with torch.no_grad():
        for name, module in model.named_modules():
            if isinstance(module, nn.Linear):
                pn = name + ".weight"
                if pn in masks:
                    mask = masks[pn]
                    w = module.weight.data
                    out_f, in_f = w.shape
                    nr, nc = out_f // TILE_R, in_f // TILE_C
                    for i in range(nr):
                        for j in range(nc):
                            if mask[i, j]:
                                w[i*TILE_R:(i+1)*TILE_R, j*TILE_C:(j+1)*TILE_C] = 0
    return masks

def restore_mlp_weights():
    with torch.no_grad():
        for name, module in model.named_modules():
            if isinstance(module, nn.Linear):
                pn = name + ".weight"
                if pn in ORIGINAL_MLP_WEIGHTS:
                    module.weight.data.copy_(ORIGINAL_MLP_WEIGHTS[pn].to(module.weight.device))

## 5. Calibrate Taylor importance once (on dense)

In [9]:
print("Taylor calibration on dense model...")
importance_taylor = compute_taylor_importance(model, cal_encodings, tag="dense")
print(f"calibrated {len(importance_taylor)} matrices, peak GPU {torch.cuda.max_memory_allocated()/1e9:.2f} GB")

Taylor calibration on dense model...


taylor[dense]: 100%|███████████████████████████████████████████████| 512/512 [03:21<00:00,  2.55it/s]


calibrated 108 matrices, peak GPU 7.01 GB


## 6. Phase A — Dense teacher: perplexity + F1 + cache top-K logits to disk

In [10]:
model.eval()
model.config.use_cache = True  # for generate()

print("=== Phase A — DENSE teacher ===")
t0 = time.time()
ppl_dense = eval_perplexity_and_cache(model, tokenizer, ppl_examples,
                                       cache_topk=True, k=TOP_K, tag="dense")
print(f"  dense perplexity: {ppl_dense['perplexity']:.3f}  (avg NLL {ppl_dense['nll']:.3f}, {ppl_dense['tokens']} tokens, {ppl_dense['skipped']} skipped) [{time.time()-t0:.1f}s]")

t0 = time.time()
f1_dense = eval_f1(model, tokenizer, f1_examples, tag="dense")
print(f"  dense F1: P={f1_dense['precision']:.3f}  R={f1_dense['recall']:.3f}  F1={f1_dense['f1']:.3f}  (TP={f1_dense['tp']} FP={f1_dense['fp']} FN={f1_dense['fn']}) [{time.time()-t0:.1f}s]")

=== Phase A — DENSE teacher ===


perplexity[dense]: 100%|███████████████████████████████████████████| 500/500 [00:25<00:00, 19.83it/s]


  dense perplexity: 1.725  (avg NLL 0.545, 5333 tokens, 0 skipped) [25.2s]


F1[dense]:   0%|                                                             | 0/200 [00:00<?, ?it/s]The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
F1[dense]: 100%|███████████████████████████████████████████████████| 200/200 [04:24<00:00,  1.32s/it]

  dense F1: P=0.747  R=0.522  F1=0.614  (TP=192 FP=65 FN=176) [264.2s]


In [11]:
# Save teacher cache to disk for offline distillation in notebook 10
os.makedirs("results", exist_ok=True)
torch.save({
    "cache": ppl_dense["cache"],
    "k": TOP_K,
    "few_shot_prefix": FEW_SHOT_PREFIX,
    "ppl_indices": ppl_indices.tolist(),
    "f1_indices": f1_indices.tolist(),
    "dense_ppl": ppl_dense["perplexity"],
    "dense_f1": f1_dense["f1"],
}, TEACHER_CACHE_PATH)
size_mb = os.path.getsize(TEACHER_CACHE_PATH) / 1e6
print(f"saved teacher cache to {TEACHER_CACHE_PATH} ({size_mb:.1f} MB, {len(ppl_dense['cache'])} examples × top-{TOP_K})")

# Sample predictions to sanity-check the format
print("\nfirst 5 dense predictions vs references:")
for p, r in zip(f1_dense["sample_predictions"], f1_dense["sample_references"]):
    print(f"  pred: {p}")
    print(f"  ref:  {r}")
    print()

saved teacher cache to results/teacher_conll_logits.pt (3.3 MB, 500 examples × top-64)

first 5 dense predictions vs references:
  pred: []
  ref:  [('Queens Park Rangers', 'ORG'), ('Bolton', 'ORG')]

  pred: [('Federal Reserve', 'ORG'), ('Lawrence Lindsey', 'PER'), ('CNBC', 'ORG')]
  ref:  [('Federal Reserve', 'ORG'), ('Lawrence Lindsey', 'PER'), ('U.S.', 'LOC'), ('CNBC', 'ORG'), ('U.S.', 'LOC')]

  pred: []
  ref:  [('Obilic', 'ORG')]

  pred: [('Jordan', 'PER'), ('Warner Bros.', 'ORG')]
  ref:  [('Irish', 'MISC'), ('British', 'MISC'), ('Jordan', 'PER'), ('Warner Bros', 'ORG')]

  pred: [('Israel', 'LOC'), ('Lebanon', 'LOC'), ('Hizbollah', 'MISC')]
  ref:  [('Israel', 'LOC'), ('Lebanese', 'MISC'), ('Hizbollah', 'ORG'), ('Sikhin', 'LOC')]



## 7. Phase B — Apply Taylor 10% pruning, re-evaluate as student

In [12]:
# Free the teacher cache from RAM — it's on disk now
ppl_dense["cache"] = None
gc.collect(); torch.cuda.empty_cache()

# Restore weights, apply masks
restore_mlp_weights()
masks = apply_taylor_masks_at(importance_taylor, PRUNE_SPARSITY)
actual = sum(m.sum() for m in masks.values()) / sum(m.size for m in masks.values())
print(f"Taylor {int(PRUNE_SPARSITY*100)}% mask applied  (actual sparsity {actual*100:.1f}%)")

Taylor 10% mask applied  (actual sparsity 10.0%)


In [13]:
print("=== Phase B — STUDENT (Taylor 10% pruned) ===")
t0 = time.time()
ppl_student = eval_perplexity_and_cache(model, tokenizer, ppl_examples,
                                         cache_topk=False, tag="student")
print(f"  student perplexity: {ppl_student['perplexity']:.3f}  (avg NLL {ppl_student['nll']:.3f}) [{time.time()-t0:.1f}s]")

t0 = time.time()
f1_student = eval_f1(model, tokenizer, f1_examples, tag="student")
print(f"  student F1: P={f1_student['precision']:.3f}  R={f1_student['recall']:.3f}  F1={f1_student['f1']:.3f}  (TP={f1_student['tp']} FP={f1_student['fp']} FN={f1_student['fn']}) [{time.time()-t0:.1f}s]")

# Cleanup
restore_mlp_weights()

=== Phase B — STUDENT (Taylor 10% pruned) ===


perplexity[student]: 100%|█████████████████████████████████████████| 500/500 [00:25<00:00, 19.93it/s]


  student perplexity: 2.289  (avg NLL 0.828) [25.1s]


F1[student]: 100%|█████████████████████████████████████████████████| 200/200 [06:01<00:00,  1.81s/it]


  student F1: P=0.415  R=0.356  F1=0.383  (TP=131 FP=185 FN=237) [361.1s]


## 8. Summary

In [14]:
ppl_gap = ppl_student["perplexity"] - ppl_dense["perplexity"]
f1_gap  = f1_dense["f1"] - f1_student["f1"]

print("=" * 72)
print("  Entity extraction recovery metric — Qwen 2.5 3B vs Taylor 10% pruned")
print("=" * 72)
print(f"  {'Metric':<24s} {'Dense':>12s} {'Pruned 10%':>14s} {'Gap':>10s}")
print("-" * 72)
print(f"  {'Perplexity (entity ann.)':<24s} {ppl_dense['perplexity']:>11.3f}  {ppl_student['perplexity']:>13.3f}  {ppl_gap:>+8.3f}")
print(f"  {'NLL':<24s} {ppl_dense['nll']:>11.3f}  {ppl_student['nll']:>13.3f}  {ppl_student['nll']-ppl_dense['nll']:>+8.3f}")
print(f"  {'F1 (CoNLL-2003 NER)':<24s} {f1_dense['f1']:>11.3f}  {f1_student['f1']:>13.3f}  {f1_gap:>+8.3f}")
print(f"  {'  Precision':<24s} {f1_dense['precision']:>11.3f}  {f1_student['precision']:>13.3f}")
print(f"  {'  Recall':<24s} {f1_dense['recall']:>11.3f}  {f1_student['recall']:>13.3f}")
print("=" * 72)

results = {
    "model": MODEL_NAME,
    "prune_sparsity": PRUNE_SPARSITY,
    "n_perplexity": N_PERPLEXITY_SAMPLES,
    "n_f1": N_F1_SAMPLES,
    "dense": {
        "perplexity": ppl_dense["perplexity"],
        "nll": ppl_dense["nll"],
        "f1": f1_dense["f1"],
        "precision": f1_dense["precision"],
        "recall": f1_dense["recall"],
    },
    "student_taylor10": {
        "perplexity": ppl_student["perplexity"],
        "nll": ppl_student["nll"],
        "f1": f1_student["f1"],
        "precision": f1_student["precision"],
        "recall": f1_student["recall"],
    },
    "gap": {
        "perplexity": ppl_gap,
        "f1": f1_gap,
    },
}
with open("results/conll_eval.json", "w") as f:
    json.dump(results, f, indent=2)
print("\nresults saved to results/conll_eval.json")

  Entity extraction recovery metric — Qwen 2.5 3B vs Taylor 10% pruned
  Metric                          Dense     Pruned 10%        Gap
------------------------------------------------------------------------
  Perplexity (entity ann.)       1.725          2.289    +0.564
  NLL                            0.545          0.828    +0.283
  F1 (CoNLL-2003 NER)            0.614          0.383    +0.231
    Precision                    0.747          0.415
    Recall                       0.522          0.356

results saved to results/conll_eval.json
